In [1]:
# Step 1: Import libraries
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import numpy as np
import pickle

# Step 2: Create a small demo dataset (no CSV needed)
data = {
    "text": [
        "I have a question about my last invoice",
        "My app keeps crashing when I open it",
        "I forgot my account password",
        "I was charged twice this month",
        "The website is not loading for me",
        "How can I reset my username?",
    ],
    "category": [
        "Billing", "Technical", "Account", "Billing", "Technical", "Account"
    ]
}
df = pd.DataFrame(data)

# Step 3: Train a small model for demo
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['text'])
y = df['category']
classifier = LogisticRegression()
classifier.fit(X, y)

# Step 4: Save the pipeline to pickle (optional)
pipeline = {
    "vectorizer": vectorizer,
    "classifier": classifier,
    "features": np.array(vectorizer.get_feature_names_out())
}
with open("demo_pipeline.pkl", "wb") as f:
    pickle.dump(pipeline, f)

print("✅ Pipeline saved (demo_pipeline.pkl)")

# Step 5: Function to classify new messages
def classify_message(message):
    X_input = vectorizer.transform([message])
    pred = classifier.predict(X_input)[0]

    # Find top 3 influential words
    class_index = list(classifier.classes_).index(pred)
    weights = classifier.coef_[class_index]
    row = X_input.toarray()[0]
    impact_scores = row * weights
    top_indices = np.argsort(impact_scores)[::-1]

    keywords = []
    for i in top_indices:
        if impact_scores[i] > 0:
            keywords.append(f"{pipeline['features'][i]} ({impact_scores[i]:.2f})")
        if len(keywords) >= 3:
            break

    return pred, keywords

# Step 6: Test some messages
test_messages = [
    "I can't log into my account",
    "My invoice has an extra charge",
    "The app is crashing when I open it"
]

for msg in test_messages:
    category, keywords = classify_message(msg)
    print(f"Message: {msg}")
    print(f"Predicted Category: {category}")
    print(f"Top Keywords: {', '.join(keywords)}\n")

# Step 7: Optional interactive demo
print("Type your own messages (type 'exit' to stop):")
while True:
    msg = input("Message: ")
    if msg.lower() == "exit":
        break
    category, keywords = classify_message(msg)
    print(f"Predicted Category: {category}")
    print(f"Top Keywords: {', '.join(keywords)}\n")



✅ Pipeline saved (demo_pipeline.pkl)
Message: I can't log into my account
Predicted Category: Account
Top Keywords: account (0.28)

Message: My invoice has an extra charge
Predicted Category: Billing
Top Keywords: invoice (0.35)

Message: The app is crashing when I open it
Predicted Category: Technical
Top Keywords: open (0.14), app (0.14), crashing (0.14)

Type your own messages (type 'exit' to stop):
Message: help i cant login
Predicted Category: Account
Top Keywords: 

Message: exit
